In [1]:
%pip install torch torchvision pillow


  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/c1/0d/56fb07032accbfebb4555638b6002ec5678d0942da85497e40f9405ab756/torch-2.6.0-cp39-cp39-win_amd64.whl.metadata
  Using cached torch-2.6.0-cp39-cp39-win_amd64.whl.metadata (28 kB)
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/6d/34/8b74ffdcb8e066b9abfc7a59407ae8fa9a67cb1e296f090d71bf4577f6cd/torchvision-0.21.0-cp39-cp39-win_amd64.whl.metadata
  Using cached torchvision-0.21.0-cp39-cp39-win_amd64.whl.metadata (6.3 kB)
  Obtaining dependency information for filelock from https://files.pythonhosted.org/packages/4d/36/2a115987e2d8c300a974597416d9de88f2444426de9571f4b59b2cca3acc/filelock-3.18.0-py3-none-any.whl.metadata
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/d5/f0/8fbc882ca80cf077f1b246c0e3c3465f7f415439bdea6b899f6b19f61f70/networkx-3.


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torchvision.transforms as transforms
from torchvision.datasets import Food101
from torch.utils.data import DataLoader, random_split

# Image transformations (resize, normalize, augment)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNetV2 expects 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Download Food-101
dataset = Food101(root="./data", download=True, transform=transform)

# Optional: Use a subset (e.g., 10 classes or fewer samples) to start small
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


100.0%


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Freeze all feature extractor layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier layer
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 101)
model = model.to(device)


c:\Users\samar\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\samar\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\samar/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth
100.0%


# Pretrained Model
Commented cells still have to be run. This is done using a pretrained model and then modifying the classification layer. Data is obtained from food-item dataset from torch. Hasn't been tested yet for accuracy

In [9]:
# import torch.optim as optim

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# # Training loop
# for epoch in range(5):  # adjust number of epochs
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     for inputs, labels in train_loader:
#         inputs, labels = inputs.to(device), labels.to(device)
#         optimizer.zero_grad()

#         outputs = model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()
#         _, predicted = torch.max(outputs.data, 1)
#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()

#     print(f"Epoch {epoch+1}, Loss: {running_loss:.3f}, Accuracy: {100 * correct / total:.2f}%")


In [10]:
# # Save the model
# torch.save(model.state_dict(), "food_classifier.pth")

# # Later, to load the model
# model.load_state_dict(torch.load("food_classifier.pth"))
# model.eval()  # Set to evaluation mode


In [11]:
# from PIL import Image

# def predict_image(img_path):
#     image = Image.open(img_path).convert('RGB')
#     image = transform(image).unsqueeze(0).to(device)  # Add batch dimension
#     model.eval()
#     with torch.no_grad():
#         output = model(image)
#         _, predicted = torch.max(output, 1)
#         return dataset.classes[predicted.item()]

# # Example usage
# print(predict_image("burger.jpg"))
